In [ ]:
ОБУЧЕНИЕ RESNET18

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import os

# ============================================
# 1. НАСТРОЙКИ
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

data_root = "C:\D\GazProm\nogit\Digital_core_v5_split"
batch_size = 64
num_epochs = 20
learning_rate = 0.001

# Папки для сохранения моделей
os.makedirs("models", exist_ok=True)

# ============================================
# 2. ТРАНСФОРМАЦИИ
# ============================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================
# 3. ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================

def train_resnet18(light_type):
    """
    Обучает ResNet18 на ДС или УФ
    light_type: "ДС" или "УФ"
    """
    print(f"\n{'='*60}")
    print(f"ОБУЧЕНИЕ ResNet18 для {light_type}")
    print(f"{'='*60}")
    
    # Загрузка данных
    train_dir = f"{data_root}/{light_type}/train"
    val_dir = f"{data_root}/{light_type}/val"
    
    train_dataset = ImageFolder(train_dir, transform=train_transform)
    val_dataset = ImageFolder(val_dir, transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    print(f"Классы: {train_dataset.classes}")
    print(f"Train: {len(train_dataset)} файлов, Val: {len(val_dataset)} файлов")
    
    # Модель
    from torchvision import models
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.fc.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    best_val_acc = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    
    for epoch in range(num_epochs):
        # Обучение
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            pbar.set_postfix({"Loss": f"{loss.item():.3f}", "Acc": f"{100*train_correct/train_total:.1f}%"})
        
        train_acc = 100 * train_correct / train_total
        train_loss = train_loss / len(train_loader)
        
        # Валидация
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = 100 * val_correct / val_total
        val_loss = val_loss / len(val_loader)
        
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        scheduler.step(val_loss)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"models/resnet18_{light_type}_best.pth")
            print(f"  💾 Сохранена лучшая модель (val_acc={val_acc:.2f}%)")
        
        print(f"  Epoch {epoch+1}: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%, Best={best_val_acc:.2f}%")
    
    return best_val_acc, history

# ============================================
# 4. ЗАПУСК ОБУЧЕНИЯ
# ============================================

print("НАЧАЛО ОБУЧЕНИЯ RESNET18")
print("=" * 60)

# Обучаем для ДС
acc_ds, hist_ds = train_resnet18("ДС")

# Обучаем для УФ
acc_uv, hist_uv = train_resnet18("УФ")

# ============================================
# 5. РЕЗУЛЬТАТЫ
# ============================================

print("\n" + "=" * 60)
print("ИТОГИ ОБУЧЕНИЯ RESNET18")
print("=" * 60)
print(f"ДС (дневной свет):   лучшая точность = {acc_ds:.2f}%")
print(f"УФ (ультрафиолет):   лучшая точность = {acc_uv:.2f}%")

# ============================================
# 6. ГРАФИКИ
# ============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist_ds["val_acc"], label="ДС", marker='o')
axes[0].plot(hist_uv["val_acc"], label="УФ", marker='s')
axes[0].set_xlabel("Эпоха")
axes[0].set_ylabel("Точность на валидации (%)")
axes[0].set_title("ResNet18: сравнение ДС и УФ")
axes[0].legend()
axes[0].grid()

axes[1].plot(hist_ds["train_loss"], label="ДС (train)", marker='o')
axes[1].plot(hist_uv["train_loss"], label="УФ (train)", marker='s')
axes[1].set_xlabel("Эпоха")
axes[1].set_ylabel("Потери")
axes[1].set_title("ResNet18: потери на обучении")
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.savefig("models/resnet18_comparison.png")
plt.show()

print("\n✅ Модели сохранены в папку models/")
print("   - models/resnet18_ДС_best.pth")
print("   - models/resnet18_УФ_best.pth")

In [ ]:
ДООБУЧЕНИЕ RESNET18

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
from torchvision.datasets import ImageFolder
from tqdm import tqdm

# ============================================
# 1. УСТРОЙСТВО
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

# ============================================
# 2. ПАРАМЕТРЫ
# ============================================

data_root = "C:\D\GazProm\nogit\Digital_core_v5_split"
batch_size = 64
learning_rate = 0.001

# ============================================
# 3. ТРАНСФОРМАЦИИ
# ============================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================
# 4. ЗАГРУЗКА ДАННЫХ УФ
# ============================================

train_dataset = ImageFolder(f"{data_root}/УФ/train", transform=transform)
val_dataset = ImageFolder(f"{data_root}/УФ/val", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

num_classes = len(train_dataset.classes)
print(f"Классы: {train_dataset.classes}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# ============================================
# 5. ЗАГРУЗКА МОДЕЛИ
# ============================================

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load("models/resnet18_УФ_best.pth", map_location=device))
model = model.to(device)

# Замораживаем всё, обучаем только последний слой
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

# ============================================
# 6. ОПТИМИЗАТОР
# ============================================

optimizer = optim.Adam(model.fc.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# ============================================
# 7. ДООБУЧЕНИЕ (5 ЭПОХ)
# ============================================

print("\n" + "=" * 60)
print("ДООБУЧЕНИЕ ResNet18 для УФ (ещё 5 эпох)")
print("=" * 60)

best_acc = 0

for epoch in range(5):
    model.train()
    train_correct = 0
    train_total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/5")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({"Loss": f"{loss.item():.3f}", "Acc": f"{100*train_correct/train_total:.1f}%"})
    
    train_acc = 100 * train_correct / train_total
    
    # Валидация
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "models/resnet18_УФ_best.pth")
        print(f"  💾 Новая лучшая модель (val_acc={val_acc:.2f}%)")
    
    print(f"  Epoch {epoch+1}: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%, Best={best_acc:.2f}%")

print(f"\n✅ ДООБУЧЕНИЕ ЗАВЕРШЕНО!")
print(f"Лучшая точность УФ: {best_acc:.2f}%")

In [ ]:
МЕТРИКИ RESNET18

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from torchvision import transforms, models
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

data_root = "C:\D\GazProm\nogit\Digital_core_v5_split"
batch_size = 64

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================
# ФУНКЦИЯ ДЛЯ ОЦЕНКИ МОДЕЛИ
# ============================================

def evaluate_model(light_type, model_path):
    print(f"\n{'='*60}")
    print(f"ОЦЕНКА МОДЕЛИ {light_type}")
    print(f"{'='*60}")
    
    # Загрузка данных
    val_dataset = ImageFolder(f"{data_root}/{light_type}/val", transform=transform)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    class_names = val_dataset.classes
    num_classes = len(class_names)
    print(f"Классы: {class_names}")
    print(f"Val размер: {len(val_dataset)}")
    
    # Загрузка модели
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    
    # Сбор предсказаний
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    # Метрики
    print("\nCLASSIFICATION REPORT:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    
    print(f"\nF1-score (macro): {f1_macro:.4f}")
    print(f"F1-score (weighted): {f1_weighted:.4f}")
    print(f"Precision (macro): {precision:.4f}")
    print(f"Recall (macro): {recall:.4f}")
    
    # Матрица ошибок
    cm = confusion_matrix(all_labels, all_preds)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix - ResNet18 {light_type}')
    plt.xticks(rotation=45)
    plt.yticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_{light_type}.png")
    plt.show()
    
    # Точность по классам
    print("\nТОЧНОСТЬ ПО КЛАССАМ:")
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    for i, class_name in enumerate(class_names):
        accuracy = cm_normalized[i, i] * 100
        print(f"  {class_name:20s}: {accuracy:.2f}%")
    
    # Топ ошибок
    print("\nТОП-5 ЧАСТЫХ ОШИБОК:")
    errors = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i, j] > 0:
                errors.append((class_names[i], class_names[j], cm[i, j]))
    
    errors.sort(key=lambda x: x[2], reverse=True)
    for i, (true, pred, count) in enumerate(errors[:5]):
        print(f"  {i+1}. {true} → {pred}: {count} раз")
    
    accuracy = np.sum(np.diag(cm)) / np.sum(cm) * 100
    print(f"\nОБЩАЯ ТОЧНОСТЬ: {accuracy:.2f}%")

# ============================================
# ЗАПУСК ДЛЯ ОБОИХ ТИПОВ
# ============================================

# Оцениваем ДС модель
evaluate_model("ДС", "models/resnet18_ДС_best.pth")

# Оцениваем УФ модель
evaluate_model("УФ", "models/resnet18_УФ_best.pth")